In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from datetime import datetime
from wavelengths import *

In [2]:
def get_bias(shifts):
    from scipy.special import voigt_profile
    from modulation import modulation_matrix
    from fit_pv import fit_pv

    q_B = 299792458 / 6173.341 * 0.231
    sigma = 0.043
    gamma = 0.053

    D = np.linalg.inv(modulation_matrix())
    F = []
    for shift in shifts:
        f = voigt_profile(shift, sigma, gamma)
        F += [f]
    F = D @ np.array(F)

    shift_ = np.round(np.mean(shifts, axis=0), 3)
    bias = fit_pv(F[0] + F[3], shift_)[0] - fit_pv(F[0] - F[3], shift_)[0]
    bias *= q_B
    return bias

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/data/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T014503_V202604260832C_0644090501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T080003_V202606261130C_0644090503.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T110003_V202607131830C_0644090504.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T140003_V202607131930C_0644090505.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T170003_V202607131930C_0644090506.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T200003_V202607151630C_0644090507.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T230003_V202607131730C_0644090508.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T014503_V202604260832C_0644100501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T050003_V202604260932C_0644

In [4]:
file = files[0]

with fits.open(file) as hdul:
    header = hdul[0].header
    fg_data = hdul['PHI_FITS_FG_settings'].data
    pmp_data = hdul['PHI_FITS_PMP_settings'].data

fg_times = np.array([datetime.fromisoformat(temp) for temp in fg_data['RecordTime']])
fg_voltages = fg_data['PHI_FG_voltage'].astype(float)

pmp_times = np.array([datetime.fromisoformat(temp) for temp in pmp_data['RecordTime']])
pmp_voltages1 = pmp_data['PHI_PMP_FDT_voltage1'].astype(float)
pmp_voltages2 = pmp_data['PHI_PMP_FDT_voltage2'].astype(float)
pmp_states = pmp_data['PHI_PMP_FDT_state']

In [5]:
get_mean_voltages(fg_data, pmp_data)

array([[-4.50000000e+02, -7.33333333e-01,  1.96733333e+02,
         3.93333333e+02,  5.92000000e+02,  7.91733333e+02],
       [-4.49866667e+02, -5.33333333e-01,  1.96400000e+02,
         3.93600000e+02,  5.92333333e+02,  7.91466667e+02],
       [-4.50133333e+02, -6.66666667e-01,  1.96866667e+02,
         3.93600000e+02,  5.92733333e+02,  7.91333333e+02],
       [-4.50000000e+02, -6.66666667e-01,  1.96533333e+02,
         3.93933333e+02,  5.92200000e+02,  7.90866667e+02]])

In [5]:
get_wavelengths(header, fg_data, pmp_data, update_header=True)

array([[6173.3835275 , 6173.54135488, 6173.61072492, 6173.6797905 ,
        6173.7495821 , 6173.81974842],
       [6173.38357434, 6173.54142514, 6173.61060782, 6173.67988418,
        6173.7496992 , 6173.81965474],
       [6173.38348066, 6173.5413783 , 6173.61077176, 6173.67988418,
        6173.74983972, 6173.8196079 ],
       [6173.3835275 , 6173.5413783 , 6173.61065466, 6173.68000128,
        6173.74965236, 6173.81944396]])

In [6]:
header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    3 / number of data axes                            
NAXIS1  =                 1024 / length of data axis 1                          
NAXIS2  =                 1024 / length of data axis 2                          
NAXIS3  =                   24 / length of data axis 3                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LONGSTRN= 'OGIP 1.0'           / The HEASARC Long String Convention may be used.
COMMENT   This FITS file may contain long string keyword values that are        
COMMENT   continued over multiple keywords.  The HEASARC convention uses the &  
COMMENT   character at the e

In [5]:
voltages = get_mean_voltages(fg_voltages, fg_times, pmp_times)
shifts = voltages * 3.513e-4 + (header['FGOV1PT1'] - 61) * 4.01225e-2 - header['OBS_VR'] * 6173.341 / 299792458
bias = get_bias(shifts)
bias

np.float64(-1.0172404213289052)

In [10]:
shifts = []

for file in files[:10]:
    with fits.open(file) as hdul:
        fg_data = hdul['PHI_FITS_FG_settings'].data
        fg_header = hdul['PHI_FITS_FG_settings'].header
        pmp_data = hdul['PHI_FITS_PMP_settings'].data
        pmp_header = hdul['PHI_FITS_PMP_settings'].header

    shifts_ = get_wavelengths(header, fg_data, pmp_data, update_header=False)
    shifts_ -= header['OBS_VR'] * 6173.341 / 299792458
    shifts += [shifts_]

shifts = np.array(shifts)
#np.savez('shifts.npz', shifts=shifts)

In [9]:
shifts_

array([[6173.3835275 , 6173.54135488, 6173.61072492, 6173.6797905 ,
        6173.7495821 , 6173.81974842],
       [6173.38357434, 6173.54142514, 6173.61060782, 6173.67988418,
        6173.7496992 , 6173.81965474],
       [6173.38348066, 6173.5413783 , 6173.61077176, 6173.67988418,
        6173.74983972, 6173.8196079 ],
       [6173.3835275 , 6173.5413783 , 6173.61065466, 6173.68000128,
        6173.74965236, 6173.81944396]])

In [27]:
shifts = np.load('shifts.npz')['shifts']
shifts.shape

In [11]:
biases = np.array([get_bias(shifts_) for shifts_ in shifts])

In [12]:
from scipy.ndimage import gaussian_filter

plt.figure(figsize=(10,8))
plt.plot(biases, lw=0.5)
plt.plot(gaussian_filter(biases, 5), 'black', lw=2)
plt.ylim(-2,2)
plt.grid(True)
plt.tight_layout()

In [7]:
np.searchsorted([0,1,1,2], 1, side='right')

np.int64(3)